## Document term matrices for each of the 5 datasets

In [ ]:
import json
import pickle
import numpy as np
import os
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import load_npz
from prettytable import PrettyTable

In [ ]:
import sys
import plotly.graph_objects as go


In [ ]:

from scipy.sparse import load_npz, issparse
from sklearn.decomposition import NMF


In [ ]:
import pandas as pd

Using CountVectorizer to build DTMs in one step and creating vocabulary files to do the mapping between entries and terms - ignoring overly common or overly rare tokens - calculating sparsity of DTMs

In [ ]:
#setting up folder paths 
root_folder = "../articles_raw_proc_data"
matrices_folder = "../articles_raw_proc_data/matrices"
models_folder = "../models"
train_file_path = "../articles_raw_proc_data/processed_data/guardian_train.json"

#Load articles from train set
train_file = open(train_file_path, "r", encoding="utf-8")
train = json.load(train_file)
train_file.close()
print("total training articles loaded: " + str(len(train)))

#Train set contains fields for each of the datasets
fields_to_process = ["text_all", "text_all_lemma", "text_nouns", "text_nouns_lemma", "text_content_stem"]

#looping over each field one at a time
for field in fields_to_process:
    #collecting the concatenated text for this field from every article
    texts_for_this_field = []
    for article in train:
        one_text = article[field]
        texts_for_this_field.append(one_text)

    #Building vectorizer
    vectorizer = CountVectorizer(min_df=2, max_df=0.95, max_features=None)

    #Applying vectorizer to dataset to obtain document-term matrix
    dtm = vectorizer.fit_transform(texts_for_this_field)

    #Separate file for vocabulary map columns of DTM to actual words
    vocab = vectorizer.get_feature_names_out()

    #building save paths for the matrix and vocabulary files
    dtm_save_path   = matrices_folder + "/dtm_" + field + ".npz"
    vocab_save_path = matrices_folder + "/vocab_" + field + ".npy"
    vec_save_path   = models_folder   + "/vectorizer_" + field + ".pkl"

    #saving the sparse matrix as .npz (a compressed format)
    save_npz(dtm_save_path, dtm)
    print("  saved dtm to: " + dtm_save_path)

    #saving the vocabulary array with numpy
    np.save(vocab_save_path, vocab)
    print("  saved vocab to: " + vocab_save_path)

    #saving the fitted vectorizer with pickle so we can reload it later
    #we need to open in binary write mode "wb" for pickle — not sure why text mode doesn't work
    vec_file = open(vec_save_path, "wb")
    pickle.dump(vectorizer, vec_file)
    vec_file.close()
    print("  saved vectorizer to: " + vec_save_path)

    #pulling out shape info for the summary print
    #dtm.shape is a tuple like (num_docs, num_words)
    num_docs  = dtm.shape[0]
    num_words = dtm.shape[1]

    #calculating sparsity 
    #nnz = number of non-zero entries in the sparse matrix
    total_cells   = num_docs * num_words
    nonzero_cells = dtm.nnz
    zero_cells    = total_cells - nonzero_cells

    sparsity_fraction = zero_cells / total_cells
    sparsity_percent = round(sparsity_fraction * 100, 4)
    sparsity_string  = str(sparsity_percent) + "%"

    #printing the summary line for this field
    print("  field=" + field + "  docs=" + str(num_docs) + "  vocab=" + str(num_words) + "  sparsity=" + sparsity_string)
    print("  ----")

#Confirming save
print("saved dtms to folder:        " + matrices_folder)
print("saved vectorizers to folder: " + models_folder)


## NNSVD-LRC initialization 



Scripts for NNSVD-LRC are ported from MATLAB code by Claude Sonnet 4.6

In [ ]:
#importing from local file
current_folder = os.getcwd()
sys.path.insert(0, current_folder)

#importing the custom functions from the py files
from nnsvdlrc import nnsvdlrc
from eval_h_stability import top_n_indices, pairwise_ats

#DTM of datasets are stored as sparse .npz files 
#Converting integers to decimals which the NMF algorithm needs to work properly
matrices_folder = "articles_raw_proc_data/matrices"
dtm_file_path = matrices_folder + "/dtm_text_all_lemma.npz"
dtm = load_npz(dtm_file_path).astype(float)

#setting rank, number of runs of the initizalization scheme and top words used for ATS
r = 13
n_runs = 1
top = 25
print("parameters set: r=" + str(r) + ", n_runs=" + str(n_runs) + ", top=" + str(top))

#setting up empty lists to store results from each run
all_h_matrices = []       
all_w_matrices = []       
all_final_errors = []     
all_iteration_counts = [] 
all_rankings = []         

for i in range(n_runs):
 
    #run the actual factorization — returns W, H, and provide
    #variables for other output from nnsvd-lrc script
    w_result, h_result, unused1, unused2, error_history = nnsvdlrc(dtm, r)

    #store the H matrix for this run
    all_h_matrices.append(h_result)

    #store the W matrix for this run
    all_w_matrices.append(w_result)

    #the final error is the last value in the error history list
    final_error_this_run = error_history[-1]
    all_final_errors.append(final_error_this_run)

    #count how many iterations it took
    number_of_iters = len(error_history)
    all_iteration_counts.append(number_of_iters)

    #get the top-n word indices for each topic in this run
    rankings_this_run = top_n_indices(h_result, top)
    all_rankings.append(rankings_this_run)

print("all runs finished!")
print("total runs completed: " + str(len(all_final_errors)))

#best = the run with the lowest reconstruction error
#we need to find the index of the minimum value in all_final_errors manually
#np.argmin does this but returns a numpy integer, int() converts it to a normal python int
best_run_index = int(np.argmin(all_final_errors))
print("best run index: " + str(best_run_index) + " with error: " + str(round(all_final_errors[best_run_index], 6)))

#save the best H and W matrices as .npy files
best_h = all_h_matrices[best_run_index]
best_w = all_w_matrices[best_run_index]

h_save_path = models_folder + "/H_init_text_all_lemma13.npy"
w_save_path = models_folder + "/W_init_text_all_lemma13.npy"

#saving to models folder
models_folder = "../models"
np.save(h_save_path, best_h)
np.save(w_save_path, best_w)
print("saved H matrix to: " + h_save_path)
print("saved W matrix to: " + w_save_path)

#comparing runs wit
#we need all pairs of run indices 

#building the list of all pairs manually since i can't use combinations()
#first collect all run indices
all_run_indices = []
for idx in range(n_runs):
    all_run_indices.append(idx)

#now find all unique pairs by looping through twice
#the j > i condition makes sure we don't compare a run to itself
#and don't repeat the same pair in reverse order
all_pairs = []
for first_idx in range(len(all_run_indices)):
    for second_idx in range(len(all_run_indices)):
        #only keep pairs where second is greater than first to avoid duplicates
        if second_idx > first_idx:
            this_pair = [all_run_indices[first_idx], all_run_indices[second_idx]]
            all_pairs.append(this_pair)

print("number of pairs to compare: " + str(len(all_pairs)))

#set up table with column names
#align sets text alignment per column
results_table = PrettyTable(["pair", "ATS", "per-topic matches"])
results_table.align["per-topic matches"] = "l"
results_table.align["ATS"] = "r"

#loop through every pair and calculate ATS
for pair in all_pairs:
    #get the two run indices from the pair
    run_i = pair[0]
    run_j = pair[1]

    #pairwise_ats returns the overall score, the matched topic pairs, and a similarity matrix S
    ats_score, matched_topics, similarity_matrix = pairwise_ats(all_rankings[run_i], all_rankings[run_j])

    #building the per-topic match string 
    match_string_parts = []
    for match in matched_topics:
        topic_from_i = match[0]
        topic_from_j = match[1]

        #get the similarity score for this pair from the matrix
        #adding 1 because topic numbering starts at 1 but indices start at 0
        similarity_score = similarity_matrix[topic_from_i, topic_from_j]
        similarity_rounded = round(float(similarity_score), 2)

        #build the label for this match 
        this_match_label = "T" + str(topic_from_i + 1) + "->T" + str(topic_from_j + 1) + "(" + str(similarity_rounded) + ")"
        match_string_parts.append(this_match_label)

    #join all match labels with double space between them
    full_match_string = ""
    for part_index in range(len(match_string_parts)):
        full_match_string = full_match_string + match_string_parts[part_index]
        #add double space between items but not after the last one
        if part_index < len(match_string_parts) - 1:
            full_match_string = full_match_string + "  "

    #format the pair label 
    pair_label = str(run_i + 1) + " vs " + str(run_j + 1)

    #round ats score for display
    ats_score_rounded = round(float(ats_score), 4)

    #add this row to the table
    results_table.add_row([pair_label, str(ats_score_rounded), full_match_string])

#print the final summary table
print("")
print("All-pairs ATS  (top " + str(top) + " terms, r=" + str(r) + ")")
print(results_table)

Spot-checking H after NNSVD-LRC

In [ ]:
models_folder = "../models"

#loading the vectorizer
vectorizer_path = models_folder + "/vectorizer_text_all_lemma.pkl"
vectorizer_file = open(vectorizer_path, "rb")
vectorizer = pickle.load(vectorizer_file)
vectorizer_file.close()

#get_feature_names_out() returns all the words the vectorizer knows about
#wrapping in np.array so we can use index lists on it later 
vocab = np.array(vectorizer.get_feature_names_out())
print("vocabulary size: " + str(len(vocab)) + " words")

#H was saved in the previous cell - rows are topics, columns are words
h_load_path = models_folder + "/H_init_text_all_lemma14.npy"
h_matrix = np.load(h_load_path)
print("H matrix loaded, shape: " + str(h_matrix.shape))

n_top = 25
n_topics = h_matrix.shape[0]
print("number of topics: " + str(n_topics))

#each column is one topic 
#the padding makes the numbers line up
all_columns = []

for topic_idx in range(n_topics):
    print("processing topic " + str(topic_idx + 1) + " of " + str(n_topics) + "...")

    #get the weights for every word in this topic (one row of H)
    this_topic_weights = h_matrix[topic_idx]

    #argsort returns indices that would sort the array smallest to largest
    #[::-1] reverses it so we get largest first
    #[:n_top] then takes just the first n_top indices
    sorted_indices_all = np.argsort(this_topic_weights)
    sorted_indices_reversed = sorted_indices_all[::-1]
    top_indices = sorted_indices_reversed[:n_top]

    #use those indices to pull out the top words and their weights
    top_terms = vocab[top_indices]
    top_weights = this_topic_weights[top_indices]

    #finding the length of the longest word so we can pad shorter ones
    #this makes the weights line up in a column when using monospace font
    longest_word_length = 0
    for term in top_terms:
        if len(term) > longest_word_length:
            longest_word_length = len(term)

    #building the display strings for each word in this topic
    combined_strings = []
    for word_idx in range(len(top_terms)):
        this_term = top_terms[word_idx]
        this_weight = top_weights[word_idx]

        #pad the word with spaces on the right so all weights line up
        padded_term = this_term.ljust(longest_word_length)

        #format the weight to 3 decimal places
        weight_rounded = round(float(this_weight), 3)
        weight_string = "{:.3f}".format(weight_rounded)

        #combine into one display string with some space between word and number
        display_string = padded_term + "   " + weight_string
        combined_strings.append(display_string)

    #add this topic's column to the list
    all_columns.append(combined_strings)

print("all topic columns built, making the table now...")

#building the topic header labels---
#each column header says "Topic 0", "Topic 1" etc.
header_labels = []
for topic_idx in range(n_topics):
    label = "Topic " + str(topic_idx)
    header_labels.append(label)

#creating the plotly table---
#go.Table takes header and cells separately
#font family is monospace so the padding actually lines up visually
fig = go.Figure(data=[go.Table(
    header=dict(
        values=header_labels,
        fill_color="steelblue",
        font=dict(color="white", size=13, family="monospace"),
        align="center"
    ),
    cells=dict(
        values=all_columns,
        align="left",
        fill_color="white",
        font=dict(family="monospace", size=12)
    )
)])

#set a title and make the table tall enough to show all 25 rows comfortably
fig.update_layout(title="Top 25 terms per NMF topic (H_init)", height=800)

#display the interactive table
fig.show()

## NMF stability (KL + multiplicative updates)

Each run re-loads the W and H matrices initizalized previously by NNSVD-LRC, then fits NMF with those as custom init.
ATS is computed on the final H matrices across all pairs of runs.

In [ ]:
current_folder = os.getcwd()
sys.path.insert(0, current_folder)

#Importing helper functions 
from eval_h_stability import top_n_indices, pairwise_ats

matrices_folder = "../articles_raw_proc_data/matrices"
models_folder = "../models"

#loading the document-term matrix
dtm_file_path = matrices_folder + "/dtm_text_all_lemma.npz"
dtm = load_npz(dtm_file_path).astype(float)
print("dtm loaded, shape: " + str(dtm.shape))

#loading the initialisation matrices saved from the previous cell
#we use these as the starting point for the NMF instead of a random start
w_init_path = models_folder + "/W_init_text_all_lemma14.npy"
h_init_path = models_folder + "/H_init_text_all_lemma14.npy"
w_init = np.load(w_init_path)
h_init = np.load(h_init_path)
print("W_init loaded, shape: " + str(w_init.shape))
print("H_init loaded, shape: " + str(h_init.shape))

r = 13       
n_runs = 2   
top = 25     

#---empty lists to store results from each run---
all_w_matrices = []
all_h_matrices = []
all_kl_errors = []
all_frob_errors = []   #sklearn gives us this one for free
all_iteration_counts = []
all_rankings = []

#running NMF n_runs times
for i in range(n_runs):
    print("starting run " + str(i + 1) + " of " + str(n_runs) + "...")

    #setting up the NMF model
    #init="custom" means we provide our own starting W and H
    #solver="mu" = multiplicative updates — the algorithm used for KL divergence
    #beta_loss="kullback-leibler" sets the loss function
    #max_iter=500 is the maximum number of update steps before stopping
    #random_state makes results reproducible — i'm not sure why it's needed with custom init but leaving it in
    nmf_model = NMF(
        n_components=r,
        init="custom",
        solver="mu",
        beta_loss="kullback-leibler",
        max_iter=500,
        random_state=123,
    )

    #fit_transform runs the model and returns the W matrix
    #we pass .copy() so the original init matrices don't get modified between runs
    w_result = nmf_model.fit_transform(dtm, W=w_init.copy(), H=h_init.copy())

    #the H matrix is stored as .components_ after fitting
    h_result = nmf_model.components_

    #store everything from this run
    all_w_matrices.append(w_result)
    all_h_matrices.append(h_result)

    #sklearn's built-in error (frobenius norm) — stored for reference
    frob_score = nmf_model.reconstruction_err_
    all_frob_errors.append(frob_score)

    #how many iterations did it actually run
    number_of_iters = nmf_model.n_iter_
    all_iteration_counts.append(number_of_iters)

    #get the top word indices for stability comparison later
    rankings_this_run = top_n_indices(h_result, top)
    all_rankings.append(rankings_this_run)

    #progress print
    print("  run " + str(i + 1) + " done — Frob: " + str(round(frob_score, 4)) + "  iters: " + str(number_of_iters))



#selecting the best run using mean ATS---

number_of_runs = len(all_rankings)

#start with a list of zeros, one per run
mean_ats_scores = []
for idx in range(number_of_runs):
    mean_ats_scores.append(0.0)

#for each run, compare it against all other runs and average the ATS scores
for i in range(number_of_runs):
    print("computing mean ATS for run " + str(i + 1) + "...")

    #collect ATS scores against all other runs
    ats_scores_for_this_run = []
    for j in range(number_of_runs):
        #skip comparing a run to itself
        if j != i:
            ats_score_ij = pairwise_ats(all_rankings[i], all_rankings[j])[0]
            ats_scores_for_this_run.append(ats_score_ij)

    #compute the mean ATS for this run
    #summing manually then dividing 
    if len(ats_scores_for_this_run) > 0:
        total_ats = 0.0
        for score in ats_scores_for_this_run:
            total_ats = total_ats + score
        mean_ats_scores[i] = total_ats / len(ats_scores_for_this_run)
    else:
        #if there's only 1 run, there's nothing to compare against
        #just set it to 0 — this run will be selected by default
        mean_ats_scores[i] = 0.0

print("mean ATS scores per run: " + str(mean_ats_scores))

#find the highest mean ATS score
best_ats_value = mean_ats_scores[0]
for score in mean_ats_scores:
    if score > best_ats_value:
        best_ats_value = score

#find all runs that have that best score (could be a tie)
best_run_candidates = []
for idx in range(len(mean_ats_scores)):
    if mean_ats_scores[idx] == best_ats_value:
        best_run_candidates.append(idx)

#if there's a tie, pick one at random — i looked up np.random.choice for this
best_run_index = int(np.random.choice(best_run_candidates))
print("best run: " + str(best_run_index + 1) + "  (mean ATS = " + str(round(best_ats_value, 4)) + ")")

#saving the best H and W matrices
h_best_path = models_folder + "/H_best_text_all_lemma14.npy"
w_best_path = models_folder + "/W_best_text_all_lemma14.npy"
np.save(h_best_path, all_h_matrices[best_run_index])
np.save(w_best_path, all_w_matrices[best_run_index])
print("saved best H to: " + h_best_path)
print("saved best W to: " + w_best_path)

#ATS stability table across all pairs of runs---
#building all unique pairs manually (same approach as before)
all_pairs = []
for first in range(number_of_runs):
    for second in range(number_of_runs):
        if second > first:
            all_pairs.append([first, second])

print("number of pairs to compare: " + str(len(all_pairs)))

#set up the table
results_table = PrettyTable(["pair", "ATS", "per-topic matches"])
results_table.align["per-topic matches"] = "l"
results_table.align["ATS"] = "r"

#fill in one row per pair
for pair in all_pairs:
    run_i = pair[0]
    run_j = pair[1]

    ats_score, matched_topics, similarity_matrix = pairwise_ats(all_rankings[run_i], all_rankings[run_j])

    #building the match string piece by piece
    match_string_parts = []
    for match in matched_topics:
        topic_from_i = match[0]
        topic_from_j = match[1]
        this_similarity = round(float(similarity_matrix[topic_from_i, topic_from_j]), 2)
        this_label = "T" + str(topic_from_i + 1) + "->T" + str(topic_from_j + 1) + "(" + str(this_similarity) + ")"
        match_string_parts.append(this_label)

    #joining labels with double spaces
    full_match_string = ""
    for part_index in range(len(match_string_parts)):
        full_match_string = full_match_string + match_string_parts[part_index]
        if part_index < len(match_string_parts) - 1:
            full_match_string = full_match_string + "  "

    pair_label = str(run_i + 1) + " vs " + str(run_j + 1)
    ats_rounded = round(float(ats_score), 4)
    results_table.add_row([pair_label, str(ats_rounded), full_match_string])

#print the final summary
print("")
print("All-pairs ATS after NMF (KL, mu)  (top " + str(top) + " terms, r=" + str(r) + ", dtm_text_all_lemma)")
print(results_table)

## Topic-term association for full, lemmatized dataset - visual inspection of top 25 terms in H

Best run selected by highest mean ATS across all other runs. 

In [ ]:
#loading the vectorizer to get the vocabulary---
vectorizer_path = models_folder + "/vectorizer_text_all_lemma.pkl"
vectorizer_file = open(vectorizer_path, "rb")
vectorizer = pickle.load(vectorizer_file)
vectorizer_file.close()
print("vectorizer loaded")

#wrap in numpy array 
vocab = np.array(vectorizer.get_feature_names_out())
print("vocabulary size: " + str(len(vocab)))

#loading the best H matrix
h_best_path = models_folder + "/H_best_text_all_lemma14.npy"
h_best = np.load(h_best_path)
print("best H loaded, shape: " + str(h_best.shape))

#parameters
n_top = 25
n_topics = h_best.shape[0]
print("number of topics: " + str(n_topics))

#building one column per topic for the table---
all_columns = []

for topic_idx in range(n_topics):
    print("processing topic " + str(topic_idx + 1) + " of " + str(n_topics) + "...")

    #get the weights for this topic and find the top-n word indices
    this_topic_weights = h_best[topic_idx]
    sorted_indices = np.argsort(this_topic_weights)[::-1][:n_top]

    top_terms = vocab[sorted_indices]
    top_weights = this_topic_weights[sorted_indices]

    #find the longest word so we can pad shorter ones 
    longest_word = 0
    for term in top_terms:
        if len(term) > longest_word:
            longest_word = len(term)

    #build the display string for each word 
    combined_strings = []
    for word_idx in range(len(top_terms)):
        padded_term = top_terms[word_idx].ljust(longest_word)
        weight_string = "{:.3f}".format(round(float(top_weights[word_idx]), 3))
        combined_strings.append(padded_term + "   " + weight_string)

    all_columns.append(combined_strings)

#building the header labels
header_labels = []
for topic_idx in range(n_topics):
    header_labels.append("Topic " + str(topic_idx + 1))

#plotly table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=header_labels,
        fill_color="steelblue",
        font=dict(color="white", size=13, family="monospace"),
        align="center"
    ),
    cells=dict(
        values=all_columns,
        align="left",
        fill_color="white",
        font=dict(family="monospace", size=12)
    )
)])

fig.update_layout(
    title="Top 25 terms per NMF topic — best run (H_best, text_all_lemma)",
    height=800
)

fig.show()

## NPMI

Number of topics are now varied between 7 and 14 and evaluated using NPMI for topic of interest in H for N = 25

In [ ]:

matrices_folder = "../articles_raw_proc_data/matrices"
models_folder = "../models"

#loading the document-term matrix
from scipy.sparse import load_npz
dtm_file_path = matrices_folder + "/dtm_text_all_lemma.npz"
dtm = load_npz(dtm_file_path).astype(float)
print("dtm loaded, shape: " + str(dtm.shape))

#loading the vocabulary
vocab_file_path = matrices_folder + "/vocab_text_all_lemma.npy"
vocab = np.load(vocab_file_path, allow_pickle=True)
print("vocab loaded, size: " + str(len(vocab)))

#total number of documents
total_documents = dtm.shape[0]
print("total documents: " + str(total_documents))


#each entry is: number of topics, H matrix filename, which topic index to inspect
#Manual verification was needed using cells above
rank_config = [
    [7,  "H_best_text_all_lemma7.npy",   2],
    [8,  "H_best_text_all_lemma8.npy",   2],
    [9,  "H_best_text_all_lemma9.npy",   2],
    [10, "H_best_text_all_lemma10.npy",  1],
    [11, "H_best_text_all_lemma11.npy",  1],
    [12, "H_best_text_all_lemma12.npy",  1],
    [13, "H_best_text_all_lemma13.npy",  2],
    [14, "H_best_text_all_lemma14.npy",  2],
]

#how many top words per topic to use for NPMI
top_n = 25


def count_documents_with_word(dtm_matrix, word_column_index):
    #counts how many documents contain at least one occurrence of a word
    word_column = np.asarray(dtm_matrix[:, word_column_index].todense()).ravel()
    documents_with_word = int(word_column.astype(bool).sum())
    return documents_with_word

def count_documents_with_both_words(dtm_matrix, word_col_i, word_col_j):
    col_i_bool = np.asarray(dtm_matrix[:, word_col_i].todense()).ravel() > 0
    col_j_bool = np.asarray(dtm_matrix[:, word_col_j].todense()).ravel() > 0
    #& is element-wise AND — true only where both are true
    both = int(np.sum(col_i_bool & col_j_bool))
    return both

def compute_npmi_for_pair(dtm_matrix, word_i, word_j, number_of_docs):
    #compute probabilities by dividing counts by total number of documents
    prob_i  = count_documents_with_word(dtm_matrix, word_i)  / number_of_docs
    prob_j  = count_documents_with_word(dtm_matrix, word_j)  / number_of_docs
    prob_ij = count_documents_with_both_words(dtm_matrix, word_i, word_j) / number_of_docs

    #small epsilon to avoid log(0) 
    epsilon = 1e-12

    #if the two words never appear together, NPMI is -1 by convention
    if prob_ij < epsilon:
        return -1.0

    #compute PMI first, then normalise
    pmi_value = np.log(prob_ij / (prob_i * prob_j + epsilon))
    npmi_value = pmi_value / (-np.log(prob_ij + epsilon))

    return float(npmi_value)

def compute_avg_pairwise_npmi(dtm_matrix, list_of_word_indices, number_of_docs):
    #for a set of top words, compute NPMI for every pair and average them
    #this gives one coherence score for the whole topic

    #building all unique pairs manually
    all_word_pairs = []
    for first in range(len(list_of_word_indices)):
        for second in range(len(list_of_word_indices)):
            if second > first:
                all_word_pairs.append([list_of_word_indices[first], list_of_word_indices[second]])

    #if there are no pairs somehow, return nan
    if len(all_word_pairs) == 0:
        return float("nan")

    #compute NPMI for each pair and collect the scores
    all_npmi_scores = []
    for pair in all_word_pairs:
        this_npmi = compute_npmi_for_pair(dtm_matrix, pair[0], pair[1], number_of_docs)
        all_npmi_scores.append(this_npmi)

    #average the scores
    total_npmi = 0.0
    for score in all_npmi_scores:
        total_npmi = total_npmi + score
    average_npmi = total_npmi / len(all_npmi_scores)

    return float(average_npmi)

def compute_loading_coverage(h_row, top_word_indices):
    #coverage = what fraction of the total topic weight is captured by the top-n words
    #if coverage is high, the top words dominate the topic
    #if coverage is low, the topic weight is spread across many words
    total_weight = h_row.sum()
    if total_weight == 0:
        return 0.0
    top_weight = h_row[top_word_indices].sum()
    coverage = float(top_weight / total_weight)
    return coverage

#main loop: compute NPMI and coverage for each rank
all_results = []        
npmi_by_rank = {}       

for config_entry in rank_config:
    rank = config_entry[0]
    h_filename = config_entry[1]
    topic_index = config_entry[2]

    print("processing rank=" + str(rank) + " topic=" + str(topic_index) + "...")

    #load the H matrix for this rank
    h_load_path = models_folder + "/" + h_filename
    h_matrix = np.load(h_load_path)

    #get the row for the selected topic
    this_topic_row = h_matrix[topic_index]

    #find the top-n word indices for this topic
    sorted_indices = np.argsort(this_topic_row)[::-1][:top_n]
    top_indices_list = sorted_indices.tolist()

    #get the actual words
    top_words_list = []
    for idx in top_indices_list:
        top_words_list.append(vocab[idx])

    #compute loading coverage for this topic
    coverage_score = compute_loading_coverage(this_topic_row, top_indices_list)

    #compute average pairwise NPMI for the top words
    avg_npmi_score = compute_avg_pairwise_npmi(dtm, top_indices_list, total_documents)

    #store the results for this rank in a dictionary
    this_result = {}
    this_result["rank"]      = rank
    this_result["topic_idx"] = topic_index
    this_result["npmi"]      = avg_npmi_score
    this_result["coverage"]  = coverage_score
    this_result["top_words"] = top_words_list

    all_results.append(this_result)
    npmi_by_rank[rank] = avg_npmi_score

    print("  r=" + str(rank) + "  topic=" + str(topic_index) + "  NPMI=" + str(round(avg_npmi_score, 4)) + "  coverage=" + str(round(coverage_score, 3)))

print("")
print("all ranks processed!")

#building the summary table
last_col_header = "top-" + str(top_n) + " words (first 10)"

results_table = PrettyTable(["r", "topic", "avg NPMI", "coverage", last_col_header])
results_table.align[last_col_header] = "l"

for result in all_results:
    #only show the first 10 words in the table to keep it readable
    first_ten_words = []
    for word_idx in range(10):
        first_ten_words.append(result["top_words"][word_idx])

    #join the words with commas
    words_string = ""
    for word_idx in range(len(first_ten_words)):
        words_string = words_string + first_ten_words[word_idx]
        if word_idx < len(first_ten_words) - 1:
            words_string = words_string + ", "

    results_table.add_row([
        str(result["rank"]),
        str(result["topic_idx"]),
        str(round(result["npmi"], 4)),
        str(round(result["coverage"], 3)),
        words_string,
    ])

print(results_table)

npmi_save_path = models_folder + "/npmi_rank_trajectory.npy"
np.save(npmi_save_path, npmi_by_rank)


## Selecting terms based on 'lift' measure

Relevance scores of N = 25 terms for topic of interest across different values for lambda and across all other topics

In [ ]:
#loading the H matrix for rank 13
h_matrix = np.load("../NMF_code/models/H_lemma_r13.npy")
print("H matrix loaded, shape: " + str(h_matrix.shape))

#how many top words to show per topic
top_n = 25

#number of topics = number of rows in H
number_of_topics = h_matrix.shape[0]
print("number of topics: " + str(number_of_topics))
print("")

#loop through each topic and print its top words
for k in range(number_of_topics):
    #get the weights for this topic
    this_topic_row = h_matrix[k]

    #get the indices of the top-n words, sorted by weight descending
    sorted_indices = np.argsort(this_topic_row)[::-1][:top_n]

    #look up the actual words from the vocab
    top_words = []
    for idx in sorted_indices:
        top_words.append(vocab[idx])

    #join the words into a single string and print
    words_string = ", ".join(top_words)
    print("Topic " + str(k) + ": " + words_string)

Relevance scores of N = 40 terms for sibling topics across different values for lambda 

In [ ]:
#loading the H matrix
h_matrix = np.load("../NMF_code/models/H_lemma_r13.npy")
print("H matrix loaded, shape: " + str(h_matrix.shape))

k = 1
sibling_indices = [3, 5, 7]


#typicality = topic k's word weights normalised to sum to 1
#this tells us how "characteristic" each word is of topic k
k_row = h_matrix[k]
k_row_sum = k_row.sum()
phi_k = k_row / k_row_sum
print("phi_k computed, sums to: " + str(round(float(phi_k.sum()), 4)))

#sibling mass = sum of word weights across all sibling topics
#words that score high here appear a lot in the siblings
sibling_rows = h_matrix[sibling_indices]
phi_siblings = sibling_rows.sum(axis=0)

#normalise sibling mass to a distribution so it's on the same scale as phi_k
phi_siblings_sum = phi_siblings.sum()
phi_sib_norm = phi_siblings / phi_siblings_sum
print("phi_sib_norm computed, sums to: " + str(round(float(phi_sib_norm.sum()), 4)))

#---step 2: min-max scale both to [0, 1]---
#min-max scaling puts both distributions on the same scale
#so lambda can meaningfully balance between them
#formula: (x - min) / (max - min)

def minmax_scale(array):
    #scale an array so its values fall between 0 and 1
    min_val = array.min()
    max_val = array.max()
    scaled = (array - min_val) / (max_val - min_val)
    return scaled

phi_k_scaled   = minmax_scale(phi_k)
phi_sib_scaled = minmax_scale(phi_sib_norm)
print("both distributions scaled to [0, 1]")

#---step 3: lambda sweep---

top_n = 40

#list of lambda values to try — from 0.0 (pure typicality) to 1.0 (pure penalty)
lambda_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

#building a dictionary where each key is a lambda label and value is the top word list
#this gets passed to pandas to make a nice comparison table
rows_dict = {}

for lam in lambda_values:
    #compute the combined score for every word in the vocabulary
    typicality_part = (1 - lam) * phi_k_scaled
    penalty_part    = lam * phi_sib_scaled
    combined_score  = typicality_part - penalty_part

    #get the top-n word indices by score, highest first
    sorted_indices = np.argsort(combined_score)[::-1][:top_n]

    #look up the actual words
    top_words = []
    for idx in sorted_indices:
        top_words.append(vocab[idx])

    column_label = "λ=" + str(lam)
    rows_dict[column_label] = top_words

#---display as a pandas dataframe---
#each column is one lambda value, each row is one word rank
results_df = pd.DataFrame(rows_dict)
print(results_df.to_string(index=False))

Plotly table with output

In [ ]:

top_n = 40
lambda_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


words_with_loadings_dict = {}

for lam in lambda_values:
    #compute combined score for every word
    combined_score = (1 - lam) * phi_k_scaled - lam * phi_sib_scaled

    #get top-n indices sorted by score descending
    sorted_indices = np.argsort(combined_score)[::-1][:top_n]

    #build display strings 
    combined_strings = []
    for idx in sorted_indices:
        this_word = vocab[idx]
        this_loading = "{:.4f}".format(float(phi_k[idx]))
        combined_strings.append(this_word + "  (" + this_loading + ")")

    column_label = "λ=" + str(lam)
    words_with_loadings_dict[column_label] = combined_strings

df_combined = pd.DataFrame(words_with_loadings_dict)

row_colours = []
for i in range(top_n):
    if i % 2 == 0:
        row_colours.append("#f0f4f8")
    else:
        row_colours.append("white")

cell_values = []
for col in df_combined.columns:
    cell_values.append(df_combined[col].tolist())

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_combined.columns),
        fill_color="#1f3b5e",
        font=dict(color="white", size=11),
        align="center"
    ),
    cells=dict(
        values=cell_values,
        fill_color=[row_colours],
        font=dict(size=10),
        align="center",
        height=22
    )
)])

fig.update_layout(
    title="Top-" + str(top_n) + " terms by λ (topic k=" + str(k) + " vs siblings " + str(sibling_indices) + ")",
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.write_image("lambda_terms.png", width=2400, height=1200, scale=2)
print("saved as lambda_terms.png")

fig.show()